# ☀️ Pipeline de Análisis de Recurso Fotovoltaico: Graus (Huesca)

Este cuaderno documenta el flujo de trabajo (workflow) integral para el tratamiento de series temporales provenientes de una planta solar. El análisis cubre el ciclo de vida inicial del dato:
1. **Ingesta y Curación:** Carga de datos crudos y validación de variables.
2. **Feature Engineering:** Transformación de componentes temporales para análisis estacional.
3. **Persistencia de Datos:** Exportación de un dataset limpio y estandarizado para modelado predictivo (XGBoost, LSTM, etc.).

**Contexto del Proyecto:**
* **Ubicación:** Graus, Huesca (España).
* **Coordenadas:** 42.198° N, 0.349° E.
* **Fuente:** PVGIS (Photovoltaic Geography Information System).

## 1. Gestión del Entorno de Ejecución
Importamos las librerías fundamentales para el análisis de datos y visualización. Separar las dependencias al inicio es una práctica recomendada (PEP 8) que facilita la portabilidad del código.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de estilos y formatos
%matplotlib inline
pd.options.display.float_format = '{:,.2f}'.format
plt.style.use('seaborn-v0_8-muted')

## 2. Ingesta de Datos Crudos

Cada registro del dataset representa una medición con resolución horaria para las coordenadas del emplazamiento en **Huesca (42.198°, 0.349°)**. A continuación, se detalla el significado técnico de cada variable:

* **`time` (Marca temporal):** Instante exacto de la medición (normalmente en UTC). Es el eje principal para el análisis de series temporales.
* **`P` (Potencia fotovoltaica):** Potencia instantánea estimada generada por el sistema. Unidad: **Vatios (W)**.
* **`G(i)` (Irradiancia global):** Radiación solar total incidente sobre el plano inclinado del generador. Unidad: **W/m²**. Es el factor con mayor correlación con la producción.
* **`H_sun` (Altura solar):** Ángulo de elevación del sol sobre el horizonte. Unidad: **Grados (°)**. Determina la intensidad teórica y la duración del día solar.
* **`T2m` (Temperatura ambiente):** Temperatura del aire medida a 2 metros de altura sobre el nivel del suelo. Unidad: **Grados Celsius (°C)**.
* **`WS10m` (Velocidad del viento):** Rapidez del viento a una altura de 10 metros. Unidad: **Metros por segundo (m/s)**. Influye en la refrigeración y eficiencia de las celdas.
* **`Int` (Indicador de intervalo):** Variable que indica la integridad del dato o tipo de intervalo. En este dataset es una constante (`0.0`), por lo que no aporta varianza al análisis.

In [2]:
ruta_pv = "/Users/rubenartolarangel/Documents/Data Science/3er Año/Prácticas/1_Limpieza/pvgis_graus_2013_2023_hourly.csv"
df = pd.read_csv(ruta_pv, sep=";", header=0)
df.head(5)

,time,P,G(i),H_sun,T2m,WS10m,Int
0,20130101:0010,0.00,0.00,0.00,-1.33,1.79,0.00
1,20130101:0110,0.00,0.00,0.00,-1.16,1.66,0.00
2,20130101:0210,0.00,0.00,0.00,-0.34,1.45,0.00
3,20130101:0310,0.00,0.00,0.00,0.33,1.45,0.00
4,20130101:0410,0.00,0.00,0.00,0.90,1.24,0.00


## 3. Limpieza de Datos (Data Cleaning)
En esta fase, eliminamos ruido y variables que no aportan valor estadístico. La columna `Int` presenta un valor constante (`0.0`), lo que significa que no posee varianza y solo ocuparía memoria innecesaria durante el entrenamiento de modelos.

In [3]:
if 'Int' in df.columns:
    df = df.drop(columns=['Int'])
print("Columnas actuales en el dataset:", df.columns.tolist())

Columnas actuales en el dataset: ['time', 'P', 'G(i)', 'H_sun', 'T2m', 'WS10m']


## 4. Normalización Temporal y Persistencia de Datos

El formato original de PVGIS (`YYYYMMDD:HHMM`) se importa inicialmente como una cadena de texto (string), lo que impide realizar operaciones de series temporales. En esta fase final del preprocesamiento realizamos:

1. **Parsing de fechas:** Transformación de la columna `time` a objetos `datetime` de alta precisión.
2. **Indexación:** Establecemos la marca temporal como índice del DataFrame para habilitar funciones de remuestreo (resampling) y selección por periodos.
3. **Estandarización ISO 8601:** Exportamos el dataset utilizando un formato de fecha reconocido globalmente y codificación UTF-8. 

Este proceso garantiza que el archivo resultante sea totalmente compatible con modelos avanzados de Machine Learning y herramientas de BI, asegurando la integridad de la línea de tiempo.

In [4]:
# El formato PVGIS suele ser YYYYMMDD:HHMM
df["time"] = pd.to_datetime(df["time"], format="%Y%m%d:%H%M")

# 2. Definimos la ruta y guardamos
# index=False evita que se cree una columna extra con los números de fila
# sep=';' mantiene la consistencia con el archivo original de PVGIS
ruta_guardado = "pvgis_graus_2013_2023_hourly_modificado.csv"

df.to_csv(ruta_guardado, 
          index=False, 
          sep=',', 
          date_format='%Y-%m-%d %H:%M:%S',
          encoding='utf-8')

print(f"✅ Proceso completado. Dataset guardado como: {ruta_guardado}")
print(f"Total de registros procesados: {len(df)}")

✅ Proceso completado. Dataset guardado como: pvgis_graus_2013_2023_hourly_modificado.csv
Total de registros procesados: 96408
